## Imports

In [1]:
import os
import time
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta
from sklearn.preprocessing import MultiLabelBinarizer

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host= 'processing-2.processing.data.production.internal',
    # 'presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

In [4]:
print(os.getcwd())
local_dataset = '/Users/E2074/local-datasets/customer/geo-consistency'
print(local_dataset)

/Users/E2074/analytics/segmentation
/Users/E2074/local-datasets/customer/geo-consistency


## Datasets & Parameter

In [5]:
city = 'Bangalore'
date_str = '20250416'
date_obj = datetime.strptime(date_str, "%Y%m%d")

start_date = date_obj - timedelta(days=10)
end_date = date_obj

date_list = [(start_date + timedelta(days=i)).strftime("%Y%m%d") for i in range(11)]

print("start_date =", start_date.strftime("%Y%m%d"))
print("end_date =", end_date.strftime("%Y%m%d"))
print("date_range =", len(date_list))
print("list_date =", date_list)

start_date = 20250406
end_date = 20250416
date_range = 11
list_date = ['20250406', '20250407', '20250408', '20250409', '20250410', '20250411', '20250412', '20250413', '20250414', '20250415', '20250416']


In [7]:
customer_base_query = f"""

    select 
        customer_id, 
        segment  
    from 
        -- hive.experiments_internal.customer_segment_amj25
        hive.experiments_internal.customer_segmentation_amj2025
"""

# df_base = pd.read_sql(customer_base_query, conn4)

In [15]:
df_base.head(5)

,customer_id,segment
0,573f290a9b0ffc28367746a3,6-MONTHLY
1,628885aecf1a86a6f50fb4e7,8-INACTIVE
2,65166e25a436ed2db6827581,4-DAILY
3,6674e0a5c5be130a1c1e4747,6-MONTHLY
4,61cf127ffb2c50c760963ece,6-MONTHLY


In [6]:
def get_data():
    for date in date_list:
    
        query_start_time = time.time()
        base_query = f"""
        
        select
            yyyymmdd,
            customer_id,
            service_obj_service_name service_name,
            order_id,
            order_status,
            pickup_cluster,
            drop_cluster,
            pickup_location_hex_8,
            drop_location_hex_8,
            pickup_location_hex_10,
            drop_location_hex_10,
            channel_host
        from 
            orders.order_logs_snapshot
        where 
            yyyymmdd = '{date}'
            and city_name = '{city}'
            and service_obj_service_name in ('Auto', 'Link', 'CabEconomy', 'Bike Lite', 'CabPremium', 'Cab SUV', 'Auto Pet', 'Auto C2C', 'Auto NCR','Auto Pool')
        """
        
        df_base_query = pd.read_sql(base_query, connection)
        query_end_time = time.time()
        
        extract_start_time = time.time()
        df_base_query.to_parquet(local_dataset + '/ban_gc_data_dump_{}.parquet'.format(date), index=False)
        extract_end_time = time.time()
        
        query_execution_time = query_end_time - query_start_time
        local_extract_time = extract_end_time - extract_start_time
        
        print(f"Query execution time: {query_execution_time:.6f} seconds & local extract time: {local_extract_time:.6f} seconds ")


get_data()

Query execution time: 48.073216 seconds & local extract time: 1.588496 seconds 
Query execution time: 73.380164 seconds & local extract time: 2.169101 seconds 
Query execution time: 54.337673 seconds & local extract time: 1.755499 seconds 
Query execution time: 57.940032 seconds & local extract time: 2.205344 seconds 
Query execution time: 51.829940 seconds & local extract time: 1.505468 seconds 
Query execution time: 53.377577 seconds & local extract time: 1.696298 seconds 
Query execution time: 50.658307 seconds & local extract time: 1.664789 seconds 
Query execution time: 46.067507 seconds & local extract time: 1.295819 seconds 
Query execution time: 45.415295 seconds & local extract time: 1.354411 seconds 
Query execution time: 47.226251 seconds & local extract time: 1.498993 seconds 
Query execution time: 91.852930 seconds & local extract time: 2.644442 seconds 


In [10]:
parquet_files = glob.glob(os.path.join(local_dataset + "/*.parquet"))
sorted(parquet_files)

['/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20241216.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20241217.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20241218.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20241219.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20241220.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20241221.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20241222.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20241223.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20241224.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20241225.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20241226.parquet',

In [11]:
glop_parquet_files =  ['/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250217.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250218.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250219.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250220.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250221.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250222.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250223.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250224.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250225.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250226.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250227.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250228.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250301.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250302.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250303.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250304.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250305.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250306.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250307.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250308.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250309.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250310.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250311.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250312.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250313.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250314.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250315.parquet',
 '/Users/E2074/local-datasets/customer/geo-consistency/ban_gc_data_dump_20250316.parquet']

In [12]:
def get_customer():
    
    # parquet_files = glob.glob(os.path.join(local_dataset + "/*.parquet"))
    df = pd.concat([pd.read_parquet(file) for file in glop_parquet_files], ignore_index=True)    
    df = df[df['order_status'] == 'dropped']
    return df[['yyyymmdd', 'customer_id', 'order_id', 
               # 'service_name', 'order_status',
               # 'pickup_cluster', 'drop_cluster', 'pickup_location_hex_12', 'drop_location_hex_12', 'channel_host',
               'pickup_tag', 'drop_tag']]

df_data = get_customer()

In [13]:
print(df_data.shape)
df_data.head(5)

(13129733, 5)


,yyyymmdd,customer_id,order_id,pickup_tag,drop_tag
0,20250217,5f5adced529dd1f49cd8077b,67b2e6ef88a3b1678040f9f5,UNKNOWN,residential
1,20250217,64b91d432b4280040b58b981,67b23d5f13174675ce3a971d,residential,residential
2,20250217,5fe9dac34319a78d54b7ea20,67b3122176c28e6ca9dabcde,residential,transit_station
3,20250217,672dda126ee5951bdfdfc0a1,67b2f8ab443b8b695005365a,None,educational
4,20250217,615a67b177c7e6097190a1c0,67b30deef7f65953e864ae35,UNKNOWN,UNKNOWN


In [14]:
df_data.pickup_tag.unique()

array(['UNKNOWN', 'residential', None, 'transit_station',
       'household_needs', 'educational', 'office', 'leisure', 'banking',
       'health_and_personal', 'govt_amenity', 'food', 'hotels',
       'place_of_worship', 'commercial'], dtype=object)

### Analysis

transit - airport, railway stations, bus stands , metro stations <br>
office - tech parks and other offices <br>
educational - schools, college , universities <br>
food - cafe, bar , resturant <br>
govt_amenity - govt offices, police stations, courts <br>
health_and_personal - hospitals , pharmacy, clinics , spa, salon, gym <br>
hotels - hotels, lodges <br>
leisure - malls, amusement parks , gardens , theatre <br>
residential - home, pg, apartments , residential comples, gated communities <br>
banks - banks , atm <br>
household_needs - marts, daily need stores, furniture stores , super markets <br>
place_of_worship - temple , church, masjid <br>

In [15]:
df_data.head(2)

,yyyymmdd,customer_id,order_id,pickup_tag,drop_tag
0,20250217,5f5adced529dd1f49cd8077b,67b2e6ef88a3b1678040f9f5,UNKNOWN,residential
1,20250217,64b91d432b4280040b58b981,67b23d5f13174675ce3a971d,residential,residential


In [16]:
# Step 1: Melt the pickup and drop tags into one column
melted = pd.melt(df_data, id_vars='customer_id', value_vars=['pickup_tag', 'drop_tag'], 
                 value_name='tag')[['customer_id', 'tag']]
melted

,customer_id,tag
0,5f5adced529dd1f49cd8077b,UNKNOWN
1,64b91d432b4280040b58b981,residential
2,5fe9dac34319a78d54b7ea20,residential
3,672dda126ee5951bdfdfc0a1,None
4,615a67b177c7e6097190a1c0,UNKNOWN
...,...,...
26259461,63e38f99f7ddaa72cde72ed5,residential
26259462,62a5f2b24becf65c9b31a05c,residential
26259463,6308ce8f00d8327946d40c67,household_needs
26259464,65c8f6226cc0eca0f70627ce,residential


In [17]:
# Step 2: Drop nulls and 'UNKNOWN' early
melted = melted.dropna()
melted = melted[melted['tag'] != 'UNKNOWN']
melted

,customer_id,tag
1,64b91d432b4280040b58b981,residential
2,5fe9dac34319a78d54b7ea20,residential
5,6221a4bee1dc43cafc13a007,residential
6,6580487f3d1cd849dd2b8e59,residential
7,66ab2557444a8ab1a22d3ba5,residential
...,...,...
26259459,64315793e4748710e5588ab0,transit_station
26259461,63e38f99f7ddaa72cde72ed5,residential
26259462,62a5f2b24becf65c9b31a05c,residential
26259463,6308ce8f00d8327946d40c67,household_needs


In [18]:
# Step 3: Group by customer and get tag sets
grouped = melted.groupby('customer_id').agg(tags = ('tag', set), usecase_count = ('tag', 'nunique') ).reset_index()
# ['tag'].agg(lambda x: set(x))
grouped

,customer_id,tags,usecase_count
0,5737c6adddbec2203f73316a,"{place_of_worship, household_needs, residential}",3
1,5737c6aeddbec2203f733176,"{leisure, residential, govt_amenity}",3
2,5737c6b1ddbec2203f73318b,{residential},1
3,5737c6c9ddbec2203f73325d,"{office, residential}",2
4,5737c6caddbec2203f73326c,"{office, residential}",2
...,...,...,...
2676030,67d716dc810ce7da981873e7,"{office, residential}",2
2676031,67d716e4aebcc424855fa250,{residential},1
2676032,67d717937c212f7fecd46db6,{office},1
2676033,67d7179559d5bfbb2ba40d74,"{office, residential}",2


In [19]:
# del melted

In [20]:
# Step 4: Vectorize the logic
def classify_vec(tags):
    if len(tags) == 0:
        return 'single'
    elif len(tags) == 1:
        return 'single'
    elif len(tags - {'residential'}) > 1:
        return 'multiple'
    else:
        return 'single'

In [21]:
# Apply classification
grouped['geo_type'] = grouped['tags'].map(classify_vec)

In [22]:
grouped

,customer_id,tags,usecase_count,geo_type
0,5737c6adddbec2203f73316a,"{place_of_worship, household_needs, residential}",3,multiple
1,5737c6aeddbec2203f733176,"{leisure, residential, govt_amenity}",3,multiple
2,5737c6b1ddbec2203f73318b,{residential},1,single
3,5737c6c9ddbec2203f73325d,"{office, residential}",2,single
4,5737c6caddbec2203f73326c,"{office, residential}",2,single
...,...,...,...,...
2676030,67d716dc810ce7da981873e7,"{office, residential}",2,single
2676031,67d716e4aebcc424855fa250,{residential},1,single
2676032,67d717937c212f7fecd46db6,{office},1,single
2676033,67d7179559d5bfbb2ba40d74,"{office, residential}",2,single


In [23]:
mlb = MultiLabelBinarizer()

In [24]:
tag_encoded = pd.DataFrame(mlb.fit_transform(grouped['tags']), columns=mlb.classes_, index=grouped.index)

In [25]:
df_result = pd.concat([grouped, tag_encoded], axis=1)

In [26]:
df_result

,customer_id,tags,usecase_count,geo_type,banking,commercial,educational,food,govt_amenity,health_and_personal,hotels,household_needs,leisure,office,place_of_worship,residential,transit_station
0,5737c6adddbec2203f73316a,"{place_of_worship, household_needs, residential}",3,multiple,0,0,0,0,0,0,0,1,0,0,1,1,0
1,5737c6aeddbec2203f733176,"{leisure, residential, govt_amenity}",3,multiple,0,0,0,0,1,0,0,0,1,0,0,1,0
2,5737c6b1ddbec2203f73318b,{residential},1,single,0,0,0,0,0,0,0,0,0,0,0,1,0
3,5737c6c9ddbec2203f73325d,"{office, residential}",2,single,0,0,0,0,0,0,0,0,0,1,0,1,0
4,5737c6caddbec2203f73326c,"{office, residential}",2,single,0,0,0,0,0,0,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2676030,67d716dc810ce7da981873e7,"{office, residential}",2,single,0,0,0,0,0,0,0,0,0,1,0,1,0
2676031,67d716e4aebcc424855fa250,{residential},1,single,0,0,0,0,0,0,0,0,0,0,0,1,0
2676032,67d717937c212f7fecd46db6,{office},1,single,0,0,0,0,0,0,0,0,0,1,0,0,0
2676033,67d7179559d5bfbb2ba40d74,"{office, residential}",2,single,0,0,0,0,0,0,0,0,0,1,0,1,0


In [27]:
df_result.columns

Index(['customer_id', 'tags', 'usecase_count', 'geo_type', 'banking',
       'commercial', 'educational', 'food', 'govt_amenity',
       'health_and_personal', 'hotels', 'household_needs', 'leisure', 'office',
       'place_of_worship', 'residential', 'transit_station'],
      dtype='object')

In [28]:
df_result.to_csv('customer_level_single_vs_multiple_geousecase.csv', index=False, header=False)

In [30]:
df_result.geo_type.value_counts(normalize=True)

geo_type
single      0.652409
multiple    0.347591
Name: proportion, dtype: float64